In [10]:
from sklearn.metrics import pairwise_distances

from pipelines.shared.string_utils import split_camel_case
# Setup

%load_ext autoreload
%autoreload 2

import os

import torch
import numpy as np
from umap import UMAP
from torch.nn.functional import normalize
from dotenv import load_dotenv
from pandas import read_json
from sklearn.cluster import HDBSCAN
from transformers import AutoModel, AutoTokenizer

from pipelines.multi_service.mean_pooling import mean_pooling

load_dotenv()

print(f"HuggingFace cache directory is {os.environ.get('HF_HOME')}")


def should_be_skipped(class_name: str):
    return (
        class_name.endswith("Test")
        or class_name.endswith("Tests")
        or class_name.endswith("IT")
    )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
HuggingFace cache directory is /media/martin/BigData/datasets/cache/huggingface/


In [82]:
import pandas as pd

# MODEL = "google-bert/bert-base-uncased"
# MODEL = "google-bert/bert-base-cased"
GENERIC_WORDS = [
    "get",
    "set",
    "find by",
    "exists by",
    "exists",
    "update",
    "delete",
    "delete by",
    "create",
    "get all",
    "create",
    "insert",
    "insert into",
    "find all",
    "find all by",
]
MODEL = "microsoft/codebert-base"

codeBertTokenizer = AutoTokenizer.from_pretrained(MODEL)
codeBertModel = AutoModel.from_pretrained(MODEL)

# bertTokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# bertModel = AutoModel.from_pretrained("google-bert/bert-base-uncased")

# input_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service.jsonl"
# output_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service-result.jsonl"

input_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/multi-service-labeled.jsonl"
)
output_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/final-with-centroids.jsonl"
)

# input_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service.json"
# output_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service-result-codebert-combined.jsonl"

# input_file = "/media/martin/BigData/datasets/pa165-git/multi-service-filtered-labeled-evaluated.jsonl"
# output_file = "/media/martin/BigData/datasets/pa165-git/multi-service-with-distances.jsonl"

data_frame = read_json(input_file, lines=True)

# origin_property = "methodSignature"
# evaluation_property = "methodSignatures"


def extract_property(methods: list[dict[str, str]], property: str):
    return list(map(lambda m: m[property], methods))


def remove_words(text: str):
    for word in GENERIC_WORDS:
        if text.startswith(word):
            text = text.removeprefix(word).strip()

    return text


def evaluate(inputs: list[dict[str, str]]):
    mean_pooled_signature = get_embeddings(extract_property(inputs, "signature"))
    method_names = extract_property(inputs, "name")
    method_name_split = list(
        map(lambda name: remove_words(" ".join(split_camel_case(name))), method_names)
    )
    mean_names = get_embeddings(method_name_split)

    signature_normalized = normalize(mean_pooled_signature, p=2, dim=1)
    name_normalized = normalize(mean_names, p=2, dim=1)

    combined = torch.cat((signature_normalized, name_normalized), dim=-1)
    mean_pooled_numpy = combined.cpu().numpy()

    # pca = PCA(n_components=0.95, random_state=42)
    # pca_reduced = pca.fit_transform(mean_pooled_numpy)

    umap = UMAP(
        n_neighbors=5,
        min_dist=0.0,
        n_components=15,
        metric="cosine",
        random_state=42,
        init="random",
    )
    umap_reduced = umap.fit_transform(mean_pooled_numpy)
    # gap_df = calculate_gap_statistic(
    #     pca_reduced, n_refs=5, max_k=math.ceil(math.sqrt(len(inputs)))
    # )

    clustering_alg = HDBSCAN(
        min_samples=1,
        min_cluster_size=3,
        metric="cosine",
        allow_single_cluster=True,
        store_centers="medoid",
    )

    labels = clustering_alg.fit_predict(umap_reduced)

    unique_labels = set(labels)
    number_of_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    noise_count = list(labels).count(-1)
    noise_ratio = noise_count / len(labels)
    # return (get_optimal_k_programmatically(gap_df)["optimal_k"], pca_reduced)

    print(labels)
    print(clustering_alg.medoids_)
    medoids = clustering_alg.medoids_
    unique_clusters = unique_labels - {-1}
    avg_distance = 0.0
    normalized_avg_distance = 0.0
    if len(unique_clusters) > 1:
        # centroids = []
        # for label in unique_clusters:
        #     centroids.append(np.mean(umap_reduced[labels == label], axis=0))

        if len(unique_clusters) > 1:
            distances = pairwise_distances(medoids, metric="cosine")
            print(distances)
            tri_offset = np.triu_indices(distances.shape[0], k=1)
            unique_distances = distances[tri_offset]

            avg_distance = np.mean(unique_distances)

            # distances = pairwise_distances(centroids, metric="euclidean")
            # tri_offset = np.triu_indices(distances.shape[0], k=1)
            # unique_distances = distances[tri_offset]
            #
            # # 5. Calculate Metrics
            # avg_distance = np.mean(unique_distances)
            # max_distance = np.max(unique_distances)
            # normalized_avg_distance = avg_distance / max_distance

    if all(x == -1 for x in labels):
        number_of_clusters = -1

    return (
        number_of_clusters,
        noise_ratio,
        umap_reduced,
        labels,
        avg_distance,
        normalized_avg_distance,
    )


def get_embeddings(inputs, tokenizer=codeBertTokenizer, model=codeBertModel):
    tokenized_inputs = tokenizer(
        inputs, return_tensors="pt", padding=True, truncation=True, max_length=256
    )
    with torch.no_grad():
        output = model(**tokenized_inputs)
    attention_mask = tokenized_inputs["attention_mask"]
    mean_pooled = mean_pooling(output, attention_mask)
    return mean_pooled


data_frame["optimal_k"] = pd.Series()
data_frame["noise_ratio"] = pd.Series()
data_frame["cluster"] = pd.Series()
data_frame["avg_dist"] = pd.Series(dtype="float64")
results = []
for index, row in data_frame.iterrows():
    class_name = row["fullyQualifiedName"]
    methods = row["methods"]

    if len(methods) < 10:
        print(f"Skipped small class {class_name}, number of methods {len(methods)}")
        continue

    if should_be_skipped(class_name):
        print(f"Skipped Test class {class_name}, number of methods {len(methods)}")
        continue

    optimal_k, noise_ratio, pca_reduced, labels, avg_dist, normalized_avg_distance = (
        evaluate(methods)
    )

    # kmeans = KMeans(n_clusters=optimal_k, random_state=42)
    # cluster = kmeans.fit_predict(pca_reduced)
    data_frame.loc[index, "optimal_k"] = optimal_k
    data_frame.loc[index, "avg_dist"] = avg_dist * 100
    data_frame.loc[index, "noise_ratio"] = noise_ratio
    # data_frame.at[index, "cluster"] = cluster.tolist()
    data_frame.at[index, "cluster"] = labels

    results.append(f"{class_name}: {optimal_k}, {labels}, dist: {avg_dist:.4f}")

for result in results:
    print(result)

data_frame.to_json(output_file, lines=True, orient="records")

Skipped small class io.aiven.klaw.clusterapi.KafkaClusterApiApplication, number of methods 1
Skipped small class io.aiven.klaw.clusterapi.models.ClusterTopicRequest, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.ClusterSchemaRequest, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.ApiResponse, number of methods 2
Skipped small class io.aiven.klaw.clusterapi.models.ServiceAccountDetails, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.SchemaCompatibilityCheckResponse, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.TopicConfig, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.ClusterSchemaCacheResetRequest, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.LoadTopicsResponse, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.config.JwtRequestFilter, number of methods 0
Skipped small class io.aiven.klaw.clusterapi.models.AivenAclRes

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[0 0 0 0 0 1 1 1 1 1 0 0 0 0]
[[ 4.09609032  2.3872335   2.83578348  3.48891807  1.91486609  7.3935833
   0.89180213  6.41921043  4.17595482  3.53075647  2.55974913  6.51638031
   4.22281218  7.32380342  1.61292815]
 [ 2.51680112 -0.29010129  4.75836039  4.98439217  5.11956978  7.56398153
   0.41773781  3.01133871  2.03985357  1.88203907  3.52684569  3.51438451
   4.64435387  8.19583988 -1.0852598 ]]
[[0.         0.11080344]
 [0.11080344 0.        ]]
Skipped small class io.aiven.klaw.clusterapi.utils.MetricsUtils, number of methods 1
Skipped small class io.aiven.klaw.clusterapi.utils.AdminClientProperties, number of methods 0


/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[0 0 0 0 1 1 1 0 0 0]
[[ 6.98860693 -3.07591772  6.69377184 13.20521736  4.78230429 14.06496143
   0.52673686  9.55979347 12.14513206  5.62540436 10.23738003  1.33115554
   5.229321    7.07524967  5.95437002]
 [ 7.00570059 -2.32967925  6.64405441 13.88626003  5.08502531 12.73556232
   0.42557588  8.61315346 11.58621883  5.58508301 10.07354736  1.92013144
   4.19719553  6.57941341  5.40854359]]
[[0.         0.00244781]
 [0.00244781 0.        ]]


...

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[0 0 1 1 0 1 0 0 1 0 0]
[[ 2.75592756  2.71043801  3.9216609   2.52366447  4.30775595  5.39291191
   3.65885234 10.0711937   1.73156798  4.8011713   0.26304039  4.56554222
   8.64547348  4.66996098  6.92885876]
 [ 3.12334442  3.17838359  3.72657609  2.54554129  3.73415852  4.49969006
   4.5996232   8.99904251  1.51468205  6.2363534   0.43065357  5.05084419
   8.34596157  5.40618849  6.96269989]]
[[0.         0.00839085]
 [0.00839085 0.        ]]
Skipped small class io.aiven.klaw.service.MetricsControllerService, number of methods 1
Skipped small class io.aiven.klaw.service.UiControllerLoginService, number of methods 4


/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[ 0  1  1  0 -1 -1  1  4  2  2  2  0  0  2  3  1  1  0  3  2  3  3  3  4
  4 -1  2]
[[4.1809001  6.31211996 5.78824329 5.67609978 3.26116037 6.27664757
  4.94508982 2.76319909 2.96458054 1.4923712  6.16565943 4.8074007
  0.751266   5.46352386 6.0700469 ]
 [4.11912823 7.27353477 4.83613443 5.25317764 3.47559476 5.87574005
  5.19786215 3.66181016 3.79764318 1.50670493 6.7842617  4.16454554
  0.60573167 5.97381067 5.46079969]
 [3.98289251 8.27232838 3.83795333 5.2815814  3.32431483 5.81576061
  5.26329422 4.22853756 4.7400713  1.87605214 7.14150476 3.71958423
  0.78941518 6.34728146 4.90503025]
 [4.49100733 9.00210476 3.22273397 6.79120302 3.07565594 5.88657379
  5.57196808 4.77254009 5.5436511  2.74363875 6.45845652 3.6676054
  1.35797083 6.68660975 3.84472895]
 [4.27998924 8.79960728 3.20490026 6.57909155 3.03245068 6.19913626
  5.56889105 4.90244436 5.70437193 2.52930093 6.405159   3.41050649
  1.03661525 6.34488297 4.48252249]]
[[0.         0.00740795 0.0241373  0.04429768 0.04047387]

/home/martin/Projects/Master's Thesis/JENA/JENA.V2/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [94]:
# input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service-final-multiply_signature-remove-words_yes.jsonl"
input_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/final-with-centroids.jsonl"
)
# input_file = "/media/martin/BigData/datasets/pa165-git/multi-service-with-distances.jsonl"
# input_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service-result-codebert-combined.jsonl"

df = pd.read_json(input_file, lines=True)
df = df.query("optimal_k.notnull()")
more_cluster_lines = df.query("optimal_k > 2")
one_cluster_lines = df.drop(more_cluster_lines.index)


def count_rows_and_print(df: pd.DataFrame, query: str, stat_type: str):
    df_part = df.query(query)
    count = len(df_part)
    print(f"{stat_type}: {count}")
    for _, record in df_part.iterrows():
        print(record["fullyQualifiedName"])
        # print(json.dumps(record["methods"], indent=2))
    print("---")
    return count


false_negatives = count_rows_and_print(
    one_cluster_lines, "label == 1", "False negatives"
)
true_negatives = count_rows_and_print(one_cluster_lines, "label == 0", "True negatives")
false_positives = count_rows_and_print(
    more_cluster_lines, "label == 0", "False positives"
)
true_positives = count_rows_and_print(
    more_cluster_lines, "label == 1", "True positives"
)

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
accuracy = (true_positives + true_negatives) / (
    true_positives + false_positives + false_negatives + true_negatives
)
f1_score = 2 * ((precision * recall) / (precision + recall))

print(f"""
Stats
Precision: {precision}
Recall: {recall}
Accuracy: {accuracy}
F1 score: {f1_score}
-------------
""")

# eval_df = df[df["label"].notna()]
# y_true = eval_df["label"].astype(int)
# y_pred = pd.Series(0, index=eval_df.index)
# y_pred[eval_df.index.isin(more_cluster_lines.index)] = 1

# print("Stats")
# print(classification_report(y_true, y_pred, target_names=["Cohesive", "Multiservice"]))

# class_clusters = defaultdict(dict)
#
# for _, record in more_cluster_lines.iterrows():
#     k = record["optimal_k"]
#     clusters = record["cluster"]
#     methods = record["methods"]
#     class_name = record["fullyQualifiedName"]
#     cohesion = record["lcom4"]
#     noise_ratio = record["noise_ratio"]
#
#     method_clusters = {}
#     for index, cluster in enumerate(clusters):
#         cluster_number_str = str(cluster)
#         method = methods[index]
#         if cluster_number_str in method_clusters:
#             method_clusters[cluster_number_str].append(method)
#         else:
#             method_clusters[cluster_number_str] = [method]
#
#     class_clusters[class_name]["methods"] = method_clusters
#     class_clusters[class_name]["lcom4"] = cohesion
#     class_clusters[class_name]["noise_ratio"] = noise_ratio
#
# for className, metadata in class_clusters.items():
#     print(f"--- Class {className}, LCOM4: {metadata['lcom4']} ---")
#     for cluster, methods in metadata["methods"].items():
#         print(f"Cluster {cluster}: {json.dumps(methods, indent=2)}\n")
#     print()

more_cluster_lines.query("label == 0")

False negatives: 4
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UtilControllerService
---
True negatives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 10
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.r

,methods,lcom4,fullyQualifiedName,startLine,label,optimal_k,noise_ratio,cluster,avg_dist
223,"[{'name': 'findById', 'signature': 'Optional<A...",19,io.aiven.klaw.repository.AclRequestsRepo,12,0,3.0,0.0,"[1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, ...",3.557154
335,"[{'name': 'findById', 'signature': 'Optional<A...",24,io.aiven.klaw.repository.AclRepo,11,0,5.0,0.0,"[1, 3, 4, 2, 2, 3, 4, 2, 4, 1, 2, 2, 3, 2, 0, ...",7.585466
355,"[{'name': 'findById', 'signature': 'Optional<A...",11,io.aiven.klaw.repository.ActivityLogRepo,12,0,3.0,0.0,"[1, 1, 2, 2, 2, 2, 0, 0, 1, 0, 2]",0.483613
386,"[{'name': 'findById', 'signature': 'Optional<K...",15,io.aiven.klaw.repository.KwKafkaConnectorReque...,13,0,4.0,0.0,"[0, 3, 3, 3, 3, 3, 0, 1, 2, 1, 2, 2, 1, 1, 1, 0]",6.141423
395,"[{'name': 'findById', 'signature': 'Optional<T...",17,io.aiven.klaw.repository.TopicRequestsRepo,14,0,4.0,0.0,"[0, 0, 2, 2, 2, 2, 3, 2, 2, 3, 3, 3, 1, 1, 1, ...",9.513302
402,"[{'name': 'findById', 'signature': 'Optional<K...",12,io.aiven.klaw.repository.KwKafkaConnectorRepo,11,0,4.0,0.0,"[1, 1, 2, 2, 3, 2, 3, 0, 0, 0, 3, 1]",0.307438
409,"[{'name': 'findById', 'signature': 'Optional<S...",13,io.aiven.klaw.repository.SchemaRequestRepo,12,0,3.0,0.0,"[0, 1, 1, 2, 1, 1, 1, 1, 0, 2, 2, 2, 0]",0.311443
412,"[{'name': 'createConnectorRequest', 'signature...",1,io.aiven.klaw.controller.KafkaConnectController,32,0,3.0,0.0,"[0, 2, 1, 1, 1, 0, 2, 2, 1, 0, 0, 1, 0]",0.184599
413,"[{'name': 'findById', 'signature': 'Optional<T...",24,io.aiven.klaw.repository.TopicRepo,11,0,3.0,0.0,"[1, 1, 2, 2, 2, 2, 2, 2, 0, 0, 1, 0, 0, 2, 2, ...",20.372759
445,"[{'name': 'getClusterApiStatus', 'signature': ...",1,io.aiven.klaw.service.ClusterApiService,99,0,4.0,0.0,"[0, 0, 2, 2, 2, 2, 1, 1, 1, 1, 1, 3, 3, 0, 3, ...",0.769643


In [ ]:
from pipelines.shared import input_until_integer

# Labeling script

input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service.jsonl"
output_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/multi-service-labeled.jsonl"
)

df = read_json(input_file, lines=True)

df["label"] = pd.Series()

for index, row in df.iterrows():
    name = row["fullyQualifiedName"]
    methods = row["methods"]
    assigned_label = row["label"]
    lcom4 = row["lcom4"]

    label = 0
    if not should_be_skipped(name) and len(methods) > 10:
        print("-" * 10)
        print(f"Class {name} (LCOM4 {lcom4})")
        print("Methods: ")
        for method in methods:
            print(method)

        label = input_until_integer("Enter label: ")

    df.loc[index, "label"] = label
    df.to_json(output_file, lines=True, orient="records")

# Results

## Remove words: yes, names multiply: 1.3

```
False negatives: 1
io.aiven.klaw.service.UtilControllerService
---
True negatives: 10
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.ClusterApiService
---
True positives: 15
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.625
Recall: 0.9375
Accuracy: 0.7142857142857143
F1 score: 0.75
```

## Remove words: yes, names multiply: no

```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 13
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 6
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.7
Recall: 0.875
Accuracy: 0.7714285714285715
F1 score: 0.7777777777777777
-------------
```

## Remove words: no, names multiply: no

```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 10
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 9
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.6086956521739131
Recall: 0.875
Accuracy: 0.6857142857142857
F1 score: 0.717948717948718
-------------
```

## Remove words: no, names multiply: 1.3
```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 10
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.ClusterApiService
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.5833333333333334
Recall: 0.875
Accuracy: 0.6571428571428571
F1 score: 0.7000000000000001
```

## Remove words: yes, signature multiply: 1.3

```
False negatives: 1
io.aiven.klaw.clusterapi.controller.ClusterApiController
---
True negatives: 10
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 9
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.AnalyticsControllerService
---
True positives: 15
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.UtilControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.625
Recall: 0.9375
Accuracy: 0.7142857142857143
F1 score: 0.75
```
